In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l2.2 Bytes and byte-level BPE

> 🎯 **Goal:** Tokenize any text at all (no out-of-vocabulary holes) while keeping sequences short, using byte-level Byte Pair Encoding.

**Why this matters.** The character tokenizer had two weaknesses: it could not handle characters it had never seen, and it produced one token per character. Real LLMs (GPT-2, GPT-3, and friends) use byte-level BPE precisely because it fixes both. This is the tokenizer family that powers production models, so understanding it here pays off everywhere later.

**The intuition.** Start from the alphabet that can spell anything: the 256 possible byte values. Every file on earth, every emoji, every language, is just bytes, so nothing is ever out of vocabulary. Then learn shorthand. A stenographer notices that `t` and `h` appear together constantly and invents one symbol for `th`. BPE does exactly this: it repeatedly finds the most frequent adjacent pair and merges it into a single new token. After many merges, common chunks like `the` or `ing` are one token instead of three.

**The mechanics.** *BPE* stands for Byte Pair Encoding. *Byte-level* means the starting tokens are the 256 raw bytes (UTF-8), so coverage is total. *Training* the tokenizer means counting pairs in a sample and recording the merges, in order, from most to least frequent. `BPETokenizer().train(sample, 400)` learns merges until the vocabulary reaches 400 tokens. To `encode`, it applies those learned merges greedily; to `decode`, it undoes the merges back to bytes, then decodes the bytes to text. Each merge can only shorten or equal the sequence, never lengthen it.

In [ ]:
from pocketlm import BPETokenizer

corpus = open(CORPUS_PATH, encoding="utf-8").read()
bpe = BPETokenizer().train(corpus[:20000], 400)
text = "to be or not to be, that is the question"
ids = bpe.encode(text)
print("raw bytes:", len(text.encode("utf-8")), " bpe tokens:", len(ids))
print("round trip ok:", bpe.decode(ids) == text)

**What just happened.** The print compared two counts: the number of raw UTF-8 bytes in the sentence versus the number of BPE tokens after merges. The BPE count is smaller (or equal), which is the whole point: same text, fewer tokens. The round-trip check confirmed decode reproduced the original string exactly.

The assertions also encode an emoji and get it back. The emoji is not a single token; it is a handful of byte tokens stitched back together on decode. That is byte-level coverage in action: nothing is ever out of vocabulary.

⚠️ **Common pitfalls**
- Tiny or unrepresentative training samples: BPE only learns merges for pairs it actually saw. Train on 200 characters and the merges will not generalize, so real text barely compresses. Train on enough varied text for the common chunks to show up.
- Confusing tokenizer training with model training: training the BPE tokenizer just learns merge rules from frequency counts. It does not learn any meaning or do any gradient descent. That comes much later with embeddings and attention.

🏋️ **Try it yourself.** Change the merge budget from `400` to `100`, then to `800`, and rerun. Watch the BPE token count for the same sentence shrink as the budget grows. Then swap in a sentence with repeated words (like `"the the the the"`) and see how aggressively BPE collapses it.

In [ ]:
assert bpe.decode(ids) == text
assert len(ids) <= len(text.encode("utf-8"))
assert bpe.decode(bpe.encode("\U0001f600")) == "\U0001f600"